In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RecoScale-ALS-Train") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.memoryOverhead", "2g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.memory.fraction", "0.7") \
    .config("spark.memory.storageFraction", "0.2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.local.dir", "/mnt/d/spark_tmp") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.network.timeout", "300s") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()
print('Spark Started...')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 15:31:36 WARN Utils: Your hostname, JACK, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/13 15:31:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 15:31:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/13 15:31:38 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Spark Started...


In [2]:
train = spark.read.parquet("/mnt/d/Projects/recoscale/data/train.parquet")
test = spark.read.parquet("/mnt/d/Projects/recoscale/data/test.parquet")

In [3]:
user_sampling = test.select('user_idx').distinct().sample(fraction= 0.1, seed= 42)

train_sampled = train.join(user_sampling, on= 'user_idx')
test_sampled = test.join(user_sampling, on= 'user_idx')

In [4]:
from pyspark.sql import functions as F

gt = test_sampled.groupBy('user_idx').agg(F.collect_set('item_idx')).alias('gt_items').cache()

users = gt.sample(fraction=0.25, seed=42).select("user_idx").cache()

In [5]:
gt.printSchema()

root
 |-- user_idx: integer (nullable = true)
 |-- collect_set(item_idx): array (nullable = false)
 |    |-- element: integer (containsNull = false)



In [ ]:
gt.orderBy('user_idx').show(truncate= False)
print(gt.count())

In [6]:
from pyspark.ml.recommendation import ALSModel

# model = ALSModel.load("/mnt/d/Projects/recoscale/models/als_(10,0.01)_model")

small_users = users.limit(5)
gt_sampled = gt.join(small_users, "user_idx")

# k = 10
# recs_small = model.recommendForUserSubset(small_users, k) \
#     .withColumn("rec_items", F.col("recommendations.item_idx"))

# df = recs_small.join(gt_sampled, "user_idx")
# df.show(truncate= False)

In [7]:
small_users.printSchema()

root
 |-- user_idx: integer (nullable = true)



In [8]:
gt_sampled.show()

+--------+---------------------+
|user_idx|collect_set(item_idx)|
+--------+---------------------+
|  222730|   [6885721, 7670764]|
|  445291|   [3458220, 5904194]|
|  487856|   [5339047, 6837958]|
|  561425|   [5129333, 7001953]|
|  573781|   [2415676, 7989383]|
+--------+---------------------+

